## 1. Mount Google Drive
Persist all journal experiment artifacts to Drive so checkpoints do not bloat the repository and work can be resumed.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/afriqa_outputs_journal
!mkdir -p /content/drive/MyDrive/afriqa_outputs_journal/journal_exports

## 2. Install Dependencies & Clone Repository

In [ ]:
import os

# Fix Colab nesting issue by always starting at root
os.chdir('/content')
if not os.path.exists('afriqa-entity-aware-qa'):
    !git clone https://github.com/OmondiKevin/afriqa-entity-aware-qa.git

os.chdir('/content/afriqa-entity-aware-qa')
!git pull
!pip install -e .
!pip install sentence-transformers sentencepiece protobuf peft

## 3. Configure Environment and Outputs
Symlink `outputs` to our dedicated Journal Drive folder. Set optimal PyTorch memory allocations.

In [ ]:
import os
import torch
from transformers import set_seed

set_seed(42)

if os.path.exists('outputs'):
    !rm -rf outputs
!ln -s /content/drive/MyDrive/afriqa_outputs_journal outputs

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1))
else:
    print('Warning: CUDA not detected.')

## 4. Download and Preprocess Dataset (If not cached)

In [ ]:
!python scripts/00_download_and_subset.py --config configs/default.yaml
!python scripts/01_prepare_qa_data.py --config configs/default.yaml

## 5. Experiment Execution Utility
Avoid overwriting valid existing outputs, dynamically checking dataset sizes.

In [ ]:
import json
import os

def get_expected_test_data():
    test_path = 'data/processed/qa_seq2seq_swa_hau_yor/test.jsonl'
    if not os.path.exists(test_path):
        return []
    data = []
    with open(test_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

EXPECTED_TEST_DATA = get_expected_test_data()
EXPECTED_COUNT = len(EXPECTED_TEST_DATA)
EXPECTED_IDS = [item.get('id', str(i)) for i, item in enumerate(EXPECTED_TEST_DATA)]

def check_artifact(filepath):
    """Returns True if valid prediction JSONL exists with exact matching test examples and order."""
    if not os.path.exists(filepath):
        return False
    if EXPECTED_COUNT == 0:
        return False
    try:
        preds = []
        with open(filepath, 'r') as f:
            for line in f:
                preds.append(json.loads(line))
        if len(preds) != EXPECTED_COUNT:
            return False
        pred_ids = [p.get('id', str(i)) for i, p in enumerate(preds)]
        if pred_ids != EXPECTED_IDS:
            return False
        return True
    except:
        pass
    return False

## 6. Run Translation Pipeline Baseline

In [ ]:
trans_pred_path = "outputs/predictions/translation_pipeline_test.jsonl"

if check_artifact(trans_pred_path):
    print("[SKIPPED] Translation Pipeline already completed.")
else:
    print("[RUNNING] Translation Pipeline inference...")
    !python scripts/02b_eval_translation_pipeline.py --config configs/default.yaml
    !python scripts/04_eval_predictions.py --config configs/default.yaml --pred_path {trans_pred_path}


## 7. Run ByT5 Low-Resource Baseline

In [ ]:
byt5_pred_path = "outputs/predictions/baseline_byt5_test.jsonl"

if check_artifact(byt5_pred_path):
    print("[SKIPPED] ByT5 Baseline already completed.")
else:
    if os.path.exists("outputs/checkpoints/baseline_byt5") and len(os.listdir("outputs/checkpoints/baseline_byt5")) > 0:
        print("[RESUMING] Resuming ByT5 Baseline training...")
    else:
        print("[RUNNING] Starting ByT5 Baseline training...")
    !python scripts/02_train_baseline_qa.py --config configs/baseline_byt5_1x.yaml
    !python scripts/04_eval_predictions.py --config configs/baseline_byt5_1x.yaml --pred_path {byt5_pred_path}


## 8. Post-Run Validation

In [ ]:
def validate_predictions():
    required_files = [trans_pred_path, byt5_pred_path]
    all_passed = True
    for f in required_files:
        if not check_artifact(f):
            print(f"❌ FAILED: {f} is missing or incomplete.")
            all_passed = False
        else:
            print(f"✅ PASSED: {f} is valid ({EXPECTED_COUNT} lines and correct ID order).")
    if all_passed:
        print("\nAll experiments completed successfully!")
    else:
        print("\nValidation failed. Check logs.")

validate_predictions()

## 9. Final Results Summary

In [ ]:
import json
import os

metrics_map = {
    'translation_pipeline': 'outputs/metrics/translation_pipeline_test_metrics.json',
    'baseline_byt5_1x': 'outputs/metrics/baseline_byt5_test_metrics.json',
}

for name, path in metrics_map.items():
    print(f'\n=== {name} ===')
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(json.dumps(data.get('overall', {}), indent=2, ensure_ascii=False))
    else:
        print(f'Missing metrics file: {path}')

## 10. Export Journal Artifacts
Copy ONLY predictions and metrics into a neat export folder in Drive so you can download them easily without pulling heavy checkpoints.

In [ ]:
import shutil

export_dir = "/content/drive/MyDrive/afriqa_outputs_journal/journal_exports"
os.makedirs(f"{export_dir}/predictions", exist_ok=True)
os.makedirs(f"{export_dir}/metrics", exist_ok=True)

if check_artifact(trans_pred_path): shutil.copy2(trans_pred_path, f"{export_dir}/predictions/")
if check_artifact(byt5_pred_path): shutil.copy2(byt5_pred_path, f"{export_dir}/predictions/")

trans_met = "outputs/metrics/translation_pipeline_test_metrics.json"
byt5_met = "outputs/metrics/baseline_byt5_test_metrics.json"

if os.path.exists(trans_met): shutil.copy2(trans_met, f"{export_dir}/metrics/")
if os.path.exists(byt5_met): shutil.copy2(byt5_met, f"{export_dir}/metrics/")

print(f"Exports saved to {export_dir}. You can download this folder directly from Google Drive!")

In [ ]:
from google.colab import runtime
print("[DONE] All experiments finished. Unassigning runtime to save A100 compute units.")
runtime.unassign()
